# PigeonGraph: Interactive Google Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hoppy-Beast/pigeongraph/blob/main/assets/pigeongraph_demo.ipynb)

Welcome to the interactive demonstration for **PigeonGraph**, a code knowledge graph and exploration engine designed for developers and AI workflows.

This notebook lets you test PigeonGraph directly in Google Colab without configuring local editors or external AI services.

### What this notebook demonstrates
1. Installing Node.js 24 LTS and PigeonGraph
2. Indexing a real target repository (the Gin Go web framework)
3. Single-turn symbol and architecture exploration (`pigeongraph explore`)
4. Python integration for automated scripts and custom tools
5. Pull request blast radius auditing (`pigeongraph audit-pr`)
6. Querying PigeonGraph's multi-package TypeScript codebase


## 1. Environment setup

PigeonGraph requires **Node.js >= 22.5.0** (Node 24 recommended for built-in `node:sqlite`).
Standard Colab instances run older Node runtimes, so we install Node 24 LTS via NodeSource.

In [ ]:
# Install Node.js 24 LTS
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1

# Confirm versions
!node -v && npm -v


## 2. Install PigeonGraph from GitHub

Clone the repository and run `npm run setup`. This automatically installs dependencies, compiles all packages, and links the global `pigeongraph` CLI executable.

In [ ]:
%%bash
# Clone repository and build/link globally
rm -rf /content/pigeongraph
git clone https://github.com/Hoppy-Beast/pigeongraph.git /content/pigeongraph
cd /content/pigeongraph
npm run setup

# Display CLI help
pigeongraph --help


## 3. Clone and initialize a target repository

To test how PigeonGraph indexes an external codebase, we clone the popular Go web framework **Gin**.

In [ ]:
%%bash
# Clone sample target repository
rm -rf /content/gin-demo
git clone --depth 1 https://github.com/gin-gonic/gin.git /content/gin-demo
cd /content/gin-demo

# Initialize PigeonGraph in the target directory
pigeongraph init


## 4. Single-turn code exploration (`pigeongraph explore`)

Traditional exploration requires repetitive grep searches and reading entire files into context.
`pigeongraph explore` returns symbol definitions, line numbers, execution flows, and blast-radius metrics in a single command.

In [ ]:
%%bash
cd /content/gin-demo
# Query the core HTTP router handler in Gin
pigeongraph explore "handleHTTPRequest"


## 5. Python programmatic usage

PigeonGraph emits structured JSON over stdout. Any Python agent, evaluation script, or automation tool can consume it directly without external libraries.

In [ ]:
import subprocess
import json
from IPython.display import display, HTML

def query_pigeongraph(repo_path: str, symbol: str) -> dict:
    res = subprocess.run(
        ["pigeongraph", "explore", symbol],
        cwd=repo_path,
        capture_output=True,
        text=True,
        check=True
    )
    return json.loads(res.stdout)

# Query the target repository
data = query_pigeongraph("/content/gin-demo", "handleHTTPRequest")

# Render formatted summary
summary = data.get("query_summary", {})
symbols = data.get("symbols", [])
blast = data.get("blast_radius", {})
spans = data.get("served_spans", [])

symbol_items = "".join([f"<li><code>{s.get('name')}</code> (<i>{s.get('kind')}</i>) in <code>{s.get('filePath')}</code></li>" for s in symbols])

html = f"""
<div style='font-family: system-ui, sans-serif; padding: 18px; border: 1px solid #d0d7de; border-radius: 8px; background: #f6f8fa;'>
  <h3 style='margin-top: 0; color: #0969da;'>PigeonGraph Exploration Result</h3>
  <p><b>Query:</b> <code>{summary.get('query')}</code> | <b>Latency:</b> <code>{summary.get('duration_ms')}ms</code> | <b>Status:</b> <code>{summary.get('epistemic_status')}</code></p>
  <p><b>Blast Radius Risk:</b> <span style='background: #dafbe1; color: #1a7f37; padding: 2px 8px; border-radius: 4px; font-weight: bold;'>{blast.get('risk_level', 'LOW')}</span> (Score: {blast.get('risk_score')})</p>
  <h4>Discovered Symbols:</h4>
  <ul>{symbol_items}</ul>
</div>
"""
display(HTML(html))

if spans:
    print("\n--- Extracted Source Code Span ---")
    print(spans[0].get("content", ""))


## 6. PR blast radius audit (`pigeongraph audit-pr`)

PigeonGraph uses invariant hashes (`H_semantic_inv`) to differentiate internal refactoring from breaking interface alterations with zero token consumption.

In [ ]:
%%bash
cd /content/pigeongraph
# Audit pull request changes against the previous commit
pigeongraph audit-pr --base HEAD~1 --head HEAD


## 7. Querying PigeonGraph's own codebase

PigeonGraph also parses TypeScript monorepos and synthesizes dynamic dispatch patterns (such as web routes and EventEmitters).

In [ ]:
# Query dynamic route synthesis in PigeonGraph
pg_data = query_pigeongraph("/content/pigeongraph", "synthesizeFrameworkRoutes")

print(f"Resolved Anchor: {pg_data['query_summary']['resolved_anchor']}")
print(f"Nodes Searched: {pg_data['query_summary']['total_graph_nodes_searched']}")
print(f"Duration: {pg_data['query_summary']['duration_ms']}ms")

for s in pg_data.get("symbols", []):
    print(f"- {s.get('name')} [{s.get('kind')}] at {s.get('filePath')}:{s.get('lineRange')}")


## Summary

In this notebook, we demonstrated:
- Installing Node.js 24 and PigeonGraph in under a minute on Ubuntu/Colab.
- Indexing and querying external projects with zero database server setup.
- Sub-millisecond single-turn exploration returning exact symbols and blast radius.
- Direct Python integration without requiring Claude or specialized agent runtimes.
- Automated PR blast radius auditing via interface invariant hashes.

For source code and documentation, visit [PigeonGraph on GitHub](https://github.com/Hoppy-Beast/pigeongraph).